In [22]:
import pandas as pd
import numpy as np
import sys   
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable

from src.config import (
    CUSTOMERS_TRAIN, LOANS_CLEAN, TRANSACTIONS_CLEAN, TRAIN_IDS, CHURN_FEATURES,DEFAULT_FEATURES,SEGMENT_FEATURES,RANDOM_STATE,
) 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score ,average_precision_score 
from sklearn.ensemble import RandomForestClassifier 
from xgboost import XGBClassifier 

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder  

In [23]:
segment_model=pd.read_parquet(SEGMENT_FEATURES)
segment_model

,customer_id,age,region,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,smartphone_user,complaints_12m,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,total_txns,total_value,active_months
0,C100000,33.0,Sindh,13.0,25-50k,31700.0,1,1,0,1,0,0.0,0,442,34,60460.0,7
1,C100002,41.0,KP,13.0,25-50k,41000.0,1,1,0,2,0,0.0,1,426,33,78460.0,11
2,C100003,23.0,Punjab,15.0,25-50k,15800.0,1,1,1,0,1,10000.0,0,399,80,159680.0,6
3,C100006,32.0,Punjab,21.0,50-100k,45500.0,2,1,0,3,1,13000.0,0,489,49,103940.0,9
4,C100007,25.0,Balochistan,17.0,50-100k,48900.0,1,1,1,0,1,19900.0,0,451,29,116010.0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,C114994,26.0,Punjab,21.0,<25k,25600.0,0,1,0,1,1,69700.0,1,499,40,77910.0,11
11996,C114995,44.0,Punjab,15.0,25-50k,24500.0,2,1,0,2,1,12500.0,1,460,23,45830.0,11
11997,C114996,43.0,Punjab,57.0,<25k,19700.0,2,1,0,2,1,23600.0,0,525,26,65280.0,7
11998,C114998,34.0,Sindh,13.0,25-50k,37400.0,0,0,2,0,0,0.0,0,370,58,114010.0,12


In [24]:
cols=["avg_monthly_inflow_pkr", "total_txns", "total_value", "active_months", "savings_balance_pkr", "has_savings", "has_insurance", "wallet_tenure_months"]

X= segment_model[cols]
X

,avg_monthly_inflow_pkr,total_txns,total_value,active_months,savings_balance_pkr,has_savings,has_insurance,wallet_tenure_months
0,31700.0,34,60460.0,7,0.0,0,0,13.0
1,41000.0,33,78460.0,11,0.0,0,1,13.0
2,15800.0,80,159680.0,6,10000.0,1,0,15.0
3,45500.0,49,103940.0,9,13000.0,1,0,21.0
4,48900.0,29,116010.0,10,19900.0,1,0,17.0
...,...,...,...,...,...,...,...,...
11995,25600.0,40,77910.0,11,69700.0,1,1,21.0
11996,24500.0,23,45830.0,11,12500.0,1,1,15.0
11997,19700.0,26,65280.0,7,23600.0,1,0,57.0
11998,37400.0,58,114010.0,12,0.0,0,0,13.0


In [25]:
X.corr()

,avg_monthly_inflow_pkr,total_txns,total_value,active_months,savings_balance_pkr,has_savings,has_insurance,wallet_tenure_months
avg_monthly_inflow_pkr,1.000000,0.191712,0.185230,0.025698,-0.021158,-0.046400,-0.024359,-0.003978
total_txns,0.191712,1.000000,0.965998,0.306948,-0.123240,-0.233082,-0.084581,0.014900
total_value,0.185230,0.965998,1.000000,0.303120,-0.121227,-0.224958,-0.079127,0.017954
active_months,0.025698,0.306948,0.303120,1.000000,-0.041610,-0.090451,-0.022675,0.039988
savings_balance_pkr,-0.021158,-0.123240,-0.121227,-0.041610,1.000000,0.508646,0.087028,-0.003469
has_savings,-0.046400,-0.233082,-0.224958,-0.090451,0.508646,1.000000,0.175114,-0.009024
has_insurance,-0.024359,-0.084581,-0.079127,-0.022675,0.087028,0.175114,1.000000,-0.011342
wallet_tenure_months,-0.003978,0.014900,0.017954,0.039988,-0.003469,-0.009024,-0.011342,1.000000


In [26]:
X=X.drop(columns=["has_savings","total_value"])
X

,avg_monthly_inflow_pkr,total_txns,active_months,savings_balance_pkr,has_insurance,wallet_tenure_months
0,31700.0,34,7,0.0,0,13.0
1,41000.0,33,11,0.0,1,13.0
2,15800.0,80,6,10000.0,0,15.0
3,45500.0,49,9,13000.0,0,21.0
4,48900.0,29,10,19900.0,0,17.0
...,...,...,...,...,...,...
11995,25600.0,40,11,69700.0,1,21.0
11996,24500.0,23,11,12500.0,1,15.0
11997,19700.0,26,7,23600.0,0,57.0
11998,37400.0,58,12,0.0,0,13.0


In [27]:
X=X.drop(columns=["has_insurance"]) 
X

,avg_monthly_inflow_pkr,total_txns,active_months,savings_balance_pkr,wallet_tenure_months
0,31700.0,34,7,0.0,13.0
1,41000.0,33,11,0.0,13.0
2,15800.0,80,6,10000.0,15.0
3,45500.0,49,9,13000.0,21.0
4,48900.0,29,10,19900.0,17.0
...,...,...,...,...,...
11995,25600.0,40,11,69700.0,21.0
11996,24500.0,23,11,12500.0,15.0
11997,19700.0,26,7,23600.0,57.0
11998,37400.0,58,12,0.0,13.0


In [28]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) 

In [29]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rows = []
km = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
segment_model["cluster"] = km.fit_predict(X_scaled)
rows.append({
    "k": 4,
    "inertia": km.inertia_,
    "silhouette": silhouette_score(X_scaled, segment_model["cluster"]),
    })

print(pd.DataFrame(rows).to_string(index=False))

 k      inertia  silhouette
 4 35717.890726    0.246895


In [30]:
segment_model["cluster"].value_counts()

cluster
1    6575
0    3198
2    1783
3     444
Name: count, dtype: int64

In [31]:
segment_model.groupby("cluster")[["avg_monthly_inflow_pkr","total_txns","active_months","savings_balance_pkr","wallet_tenure_months"]].mean().round(2)

,avg_monthly_inflow_pkr,total_txns,active_months,savings_balance_pkr,wallet_tenure_months
cluster,,,,,
0,36293.18,37.93,7.27,5869.51,25.25
1,33283.39,56.49,10.99,4635.56,30.16
2,62738.08,200.92,10.94,2585.08,28.66
3,35538.29,49.65,9.97,69713.96,28.43


In [32]:
truth = pd.read_parquet(CUSTOMERS_TRAIN)[["customer_id", "segment_true"]]
segment_model = segment_model.merge(truth, on="customer_id", how="left")

pd.crosstab(segment_model["cluster"], segment_model["segment_true"])

segment_true,borrower,merchant,payroll,saver
cluster,,,,
0,706,339,1069,1084
1,1606,413,2912,1644
2,48,1625,83,27
3,52,39,86,267
